# 01b - Auto-Ingest: Covers + Location Only

**What this notebook is for:** notebook 01 assumed you already had four separate exports (POS, reservations, weather, calendar). Most restaurants don't — usually the only reliable record is a **backlog of daily covers**. This notebook is the alternative starting point: give it just a `date, covers` file and a location, and it auto-fills weather and public holidays from free public APIs, so you land on the exact same `covers_daily_clean.csv` schema notebooks 02-06 already expect. Point it at any restaurant's data and the rest of the project runs unchanged.

**What gets auto-filled vs. what stays manual:**

| Field | Source | Automated? |
|---|---|---|
| Weather (temp, rain) | Open-Meteo historical weather API, by geocoded location | Yes |
| Public holidays | Nager.Date public holiday API, by country | Yes |
| Local events (festivals, concerts, farmers markets) | — | No reliable free universal source exists — stays a manual column, same as it would for any restaurant |
| Promos | — | Internal to the restaurant, can never be sourced externally — stays manual |

**An honest caveat before you run this:** the weather and holiday API calls need real internet access. I (Claude) built and reasoned through this code carefully against each API's published documentation, but my own sandbox has no outbound internet, so unlike every other notebook in this project, I could not personally execute these specific network calls to confirm the live response. Run it, and if a cell errors or the output looks wrong, share the error message and I'll fix it.

## Step 1: Configure — edit these for your restaurant

Only two things are strictly required: a CSV with a date column and a covers column, and a location (city + state/country works better than a full street address — this is a place-name lookup, not a precise address geocoder).

In [ ]:
# ---- EDIT THESE FOR YOUR RESTAURANT ----

INPUT_CSV = "../data/raw/bhan_covers.csv"   # Bhan covers.xlsx, converted to CSV and dropped in data/raw/
DATE_COLUMN = "date"
COVERS_COLUMN = "Total guests"   # this file tracks guests as "Total guests", not "covers"
RESERVATIONS_COLUMN = None   # set to a column name if you separately track reservations, else leave None

LOCATION = "CHANGE_ME"      # <-- set this to Bhan's city + state/country, e.g. "Austin, TX"
COUNTRY_CODE_OVERRIDE = None # e.g. "US" -- only set this if Step 2 geocodes to the wrong country

OUTPUT_CSV = "../data/processed/covers_daily_clean_auto.csv"
# ^ deliberately NOT named covers_daily_clean.csv, so this never silently overwrites an existing
#   hand-built file. Once you're happy with the result, rename it (or point notebooks 02-06 at it).

## Step 2: Load the covers backlog

Minimal validation: parse dates, sort chronologically, check for duplicate dates (same handling as notebook 01 — flag them rather than silently picking one).

In [ ]:
import pandas as pd
import numpy as np
import requests

raw = pd.read_csv(INPUT_CSV)
raw[DATE_COLUMN] = pd.to_datetime(raw[DATE_COLUMN])
raw = raw.rename(columns={DATE_COLUMN: "date", COVERS_COLUMN: "covers"})
if RESERVATIONS_COLUMN:
    raw = raw.rename(columns={RESERVATIONS_COLUMN: "reservations_covers"})

raw = raw.sort_values("date").reset_index(drop=True)

dupes = raw[raw.duplicated(subset="date", keep=False)]
if len(dupes):
    print(f"WARNING: {len(dupes)} duplicate-date rows found — review and dedupe before trusting the output:")
    print(dupes)
    raw = raw.drop_duplicates(subset="date", keep="first")

print(f"Loaded {len(raw)} rows, {raw['date'].min().date()} to {raw['date'].max().date()}")
raw.head()

## Step 3: Geocode the location

Looks up `LOCATION` against Open-Meteo's free geocoding API (no key required) and picks the best match by population. Always check the printed result — a same-named town in the wrong state is a real risk with a short place name. If it's wrong, set `COUNTRY_CODE_OVERRIDE` in Step 1, or replace `LOCATION` with something more specific.

In [ ]:
def geocode_location(query, count=5):
    resp = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": query, "count": count, "language": "en", "format": "json"},
        timeout=15,
    )
    resp.raise_for_status()
    results = resp.json().get("results", [])
    if not results:
        raise ValueError(
            f"No location found for '{query}'. Try a simpler name, e.g. 'Austin' instead of a full address."
        )
    return sorted(results, key=lambda r: r.get("population", 0), reverse=True)

candidates = geocode_location(LOCATION)
best = candidates[0]

LATITUDE = best["latitude"]
LONGITUDE = best["longitude"]
COUNTRY_CODE = COUNTRY_CODE_OVERRIDE or best["country_code"]
TIMEZONE = best.get("timezone", "auto")

print(f"Matched: {best['name']}, {best.get('admin1', '')}, {best['country']} "
      f"({LATITUDE:.3f}, {LONGITUDE:.3f}), country code {COUNTRY_CODE}")

if len(candidates) > 1:
    print("\nOther possible matches (in case this one's wrong):")
    for c in candidates[1:4]:
        print(f"  {c['name']}, {c.get('admin1','')}, {c['country']} ({c['latitude']:.3f}, {c['longitude']:.3f})")

## Step 4: Build the full date spine

Same reasoning as notebook 01: an explicit day-by-day calendar from the first to the last date in your file, so a day that's missing from your covers export becomes a visible gap (flagged `covers_missing`) instead of silently disappearing.

In [ ]:
full_range = pd.DataFrame({"date": pd.date_range(raw["date"].min(), raw["date"].max(), freq="D")})
df = full_range.merge(raw, on="date", how="left")

df["covers_missing"] = df["covers"].isna().astype(int)
print(f"Spine covers {df['date'].min().date()} to {df['date'].max().date()}, "
      f"{len(df)} days total, {df['covers_missing'].sum()} missing covers.")

## Step 5: Fetch historical weather

Open-Meteo's archive API, no key required. **One thing to expect:** its underlying data (ERA5 reanalysis) typically lags a few days behind real time, so if your covers file runs right up through yesterday, the most recent handful of days may come back with no weather yet. We interpolate small gaps the same way notebook 01 did — a reasonable estimate for a smooth variable like temperature, not for the target itself.

In [ ]:
def fetch_weather(lat, lon, start_date, end_date, timezone="auto"):
    resp = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude": lat, "longitude": lon,
            "start_date": start_date, "end_date": end_date,
            "daily": "temperature_2m_max,precipitation_sum",
            "temperature_unit": "fahrenheit",
            "precipitation_unit": "inch",
            "timezone": timezone,
        },
        timeout=30,
    )
    resp.raise_for_status()
    daily = resp.json()["daily"]
    return pd.DataFrame({
        "date": pd.to_datetime(daily["time"]),
        "temp_high_f": daily["temperature_2m_max"],
        "precip_in": daily["precipitation_sum"],
    })

weather = fetch_weather(LATITUDE, LONGITUDE, df["date"].min().date().isoformat(),
                         df["date"].max().date().isoformat(), TIMEZONE)

df = df.merge(weather, on="date", how="left")

missing_weather = df["temp_high_f"].isna().sum()
if missing_weather:
    print(f"{missing_weather} day(s) missing weather (likely too recent for the archive) — interpolating.")
    df[["temp_high_f", "precip_in"]] = df[["temp_high_f", "precip_in"]].interpolate(limit_direction="both")
else:
    print("Weather matched for every day.")

## Step 6: Fetch public holidays

Nager.Date's public holiday API, keyed off the geocoded country code, one call per calendar year your data spans. Marks each matching date with `is_holiday` and the holiday's name.

In [ ]:
def fetch_holidays(country_code, years):
    rows = []
    for year in years:
        resp = requests.get(
            f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code}",
            timeout=15,
        )
        if resp.status_code == 404:
            print(f"No holiday calendar found for country code '{country_code}', year {year} — skipping.")
            continue
        resp.raise_for_status()
        for h in resp.json():
            rows.append({"date": h["date"], "holiday_name": h["localName"]})
    return pd.DataFrame(rows)

years = range(df["date"].min().year, df["date"].max().year + 1)
holidays = fetch_holidays(COUNTRY_CODE, years)
holidays["date"] = pd.to_datetime(holidays["date"])

df = df.merge(holidays, on="date", how="left")
df["is_holiday"] = df["holiday_name"].notna().astype(int)

print(f"Matched {df['is_holiday'].sum()} holiday date(s) from {holidays['holiday_name'].nunique()} distinct holidays.")

## Step 7: Local events and promos — manual columns

No free, reliable, universal source exists for local events (farmers markets, festivals, one-off concerts), and promos are internal to the restaurant either way. Both are added here as blank columns, ready for you to fill in from your own knowledge — same as the calendar file in notebook 01, just without a pre-populated source to join against.

In [ ]:
df["is_local_event"] = 0
df["event_name"] = np.nan
df["is_promo"] = 0
df["promo_name"] = np.nan

print("Added blank is_local_event / event_name / is_promo / promo_name columns — fill these in by hand where you know of one.")

## Step 8: Calendar fields, closed-day flag, and outlier review

Derive `day_of_week`, `is_weekend`, `month`, `week_of_year` from the date, same as notebook 01. For `is_closed`: a date **present** in your file with `covers == 0` is treated as a known closure; a date **absent** from your file entirely is a genuine gap (already flagged via `covers_missing`) and left as unknown rather than assumed closed.

Outliers are **flagged for your review, not auto-corrected** — this tool doesn't know this restaurant's history the way notebook 01 could reason through a specific spike for one restaurant, so it's safer to surface anything unusual and let you decide.

In [ ]:
df["day_of_week"] = df["date"].dt.day_name()
df["is_weekend"] = df["day_of_week"].isin(["Friday", "Saturday", "Sunday"]).astype(int)
df["month"] = df["date"].dt.month
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

df["is_closed"] = ((df["covers"] == 0) & (df["covers_missing"] == 0)).astype(int)

df["z_covers"] = df.groupby("day_of_week")["covers"].transform(
    lambda s: (s - s.mean()) / s.std() if s.std() > 0 else 0
)
outliers = df.loc[df["z_covers"].abs() > 2.5, ["date", "day_of_week", "covers", "z_covers"]]
if len(outliers):
    print(f"{len(outliers)} day(s) flagged as statistical outliers (>2.5 std dev from that weekday's average) — "
          f"review before modeling, don't auto-correct blindly:")
    print(outliers.sort_values("z_covers", key=abs, ascending=False))
else:
    print("No extreme outliers flagged.")

df = df.drop(columns=["z_covers"])

## Step 9: Assemble and save

Match the exact column order notebook 01 produces, so notebooks 02-06 work against this file with zero changes.

In [ ]:
final_cols = [
    "date", "day_of_week", "is_weekend", "month", "week_of_year",
    "covers", "covers_missing",
]
if "reservations_covers" in df.columns:
    final_cols += ["reservations_covers"]
final_cols += [
    "is_holiday", "holiday_name", "is_local_event", "event_name",
    "is_promo", "promo_name", "temp_high_f", "precip_in", "is_closed",
]

final = df[final_cols].copy()
final.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(final)} rows to {OUTPUT_CSV}")
print(f"\nData quality summary:")
print(f"  Missing covers days: {final['covers_missing'].sum()}")
print(f"  Closed days: {final['is_closed'].sum()}")
print(f"  Holidays matched: {final['is_holiday'].sum()}")
print(f"  Weather coverage: {final['temp_high_f'].notna().mean():.0%}")
final.head()

## Recap and next steps

- Turned a bare `date, covers` file plus a location string into the same schema notebook 01 hand-built from four separate exports — weather and public holidays auto-filled from free APIs, keyed off a geocoded location.
- Local events and promos are intentionally left manual — no reliable free source exists for the former, and the latter is internal to the restaurant regardless.
- Outliers are flagged for review, not silently corrected — this tool doesn't have the restaurant-specific context notebook 01 used to justify a correction.

**Before moving on:** open `covers_daily_clean_auto.csv`, skim the flagged outliers and any missing-weather days, and fill in local events/promos for any dates you know about. Once you're satisfied, rename it to `covers_daily_clean.csv` (or point `INPUT_CSV`/`OUTPUT_CSV` in notebooks 02-06 at this file) and continue through the same EDA → baselines → features → model → weekly forecast pipeline already built for this project — the coefficients will refit fresh on this restaurant's own numbers, since (per your earlier question) the math genuinely differs restaurant to restaurant.